# Luong Attention from Scratch

Implementing the attention mechanism from "Effective Approaches to Attention-based Neural Machine Translation" (2015).

**Key idea:** Compute attention using the CURRENT decoder state (after LSTM), not the previous one. Simpler scoring functions: dot product, general, or concat.

**Main differences from Bahdanau:**
- Uses current hidden state s_t (not s_t-1)
- LSTM first, then attention (not attention, then LSTM)
- Simpler scoring options
- Context combined with LSTM output (not input)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Load Multi30k Dataset

In [ ]:
# Load dataset from HuggingFace
dataset = load_dataset('bentrevett/multi30k')

# Use subset for CPU training
TRAIN_SIZE = 5000
VAL_SIZE = 500

train_data = dataset['train'].select(range(TRAIN_SIZE))
val_data = dataset['validation'].select(range(VAL_SIZE))

In [ ]:
print(f'Training samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print()
print('Example:')
print(f"  EN: {train_data[200]['en']}")
print(f"  DE: {train_data[200]['de']}")

## 2. Tokenization & Vocabulary

In [ ]:
def tokenize(text):
    return text.lower().strip().split()

class Vocabulary:
    def __init__(self, name, min_freq=2):
        self.name = name
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.n_words = 4
    
    def build_vocab(self, sentences):
        counter = Counter()
        for sent in sentences:
            counter.update(tokenize(sent))     
        
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.word2idx[word] = self.n_words
                self.idx2word[self.n_words] = word
                self.n_words += 1
    
    def encode(self, sentence):
        tokens = tokenize(sentence)
        return [self.word2idx.get(w, self.word2idx['<UNK>']) for w in tokens]
    
    def decode(self, indices):
        return [self.idx2word[idx] for idx in indices if idx not in [0, 1, 2]]

# Build vocabularies
src_vocab = Vocabulary('english', min_freq=2)
trg_vocab = Vocabulary('german', min_freq=2)

src_vocab.build_vocab([ex['en'] for ex in train_data])
trg_vocab.build_vocab([ex['de'] for ex in train_data])

print(f'English vocabulary: {src_vocab.n_words} words')
print(f'German vocabulary: {trg_vocab.n_words} words')

## 3. Dataset & DataLoader

In [ ]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data, src_vocab, trg_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
    
    def __len__(self):
        return len(self.data)  
    
    def __getitem__(self, idx):
        src = self.src_vocab.encode(self.data[idx]['en']) + [self.src_vocab.word2idx['<EOS>']]
        trg = [self.trg_vocab.word2idx['<SOS>']] + self.trg_vocab.encode(self.data[idx]['de']) + [self.trg_vocab.word2idx['<EOS>']]
        return torch.tensor(src), torch.tensor(trg)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=0)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, padding_value=0)
    return src_padded, trg_padded

# Create datasets
train_dataset = TranslationDataset(train_data, src_vocab, trg_vocab)
val_dataset = TranslationDataset(val_data, src_vocab, trg_vocab)

# Create dataloaders
BATCH_SIZE = 32

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# Test
src, trg = next(iter(train_loader))
print(f'Source batch shape: {src.shape}  # [src_len, batch]')
print(f'Target batch shape: {trg.shape}  # [trg_len, batch]')

## 4. Hyperparameters

In [ ]:
# Model hyperparameters
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.1
ATTENTION_METHOD = 'general'  # 'dot', 'general', or 'concat'

# Training hyperparameters
LEARNING_RATE = 0.001
N_EPOCHS = 10
CLIP = 1.0

print('Hyperparameters set!')
print(f'  Embedding dim: {EMBEDDING_DIM}')
print(f'  Hidden dim: {HIDDEN_DIM}')
print(f'  Attention method: {ATTENTION_METHOD}')
print(f'  Epochs: {N_EPOCHS}')

## 5. Luong Attention Implementation

**Key architecture points:**

1. `Encoder` - Same as Bahdanau, returns all hidden states
2. `LuongAttention` - Three scoring methods: dot, general, concat
3. `LuongDecoder` - LSTM first, then attention (OPPOSITE of Bahdanau)
4. `Seq2SeqLuong` - Combines everything

**Scoring functions:**
```
dot:     score = s_t^T · h_i
general: score = s_t^T · W_a · h_i
concat:  score = v^T · tanh(W · [s_t; h_i])
```

**Critical difference from Bahdanau:**
- Bahdanau: attention → concat with embedding → LSTM → output
- Luong: embedding → LSTM → attention → combine output+context → predict

In [ ]:
class Encoder(nn.Module):
    """Same as Bahdanau encoder - returns all hidden states"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: [src_len, batch, hidden_dim]
        # hidden: [1, batch, hidden_dim]
        return outputs, hidden, cell

In [ ]:
class LuongAttention(nn.Module):
    """Luong attention with three scoring methods"""
    def __init__(self, hidden_dim, method='general'):
        super().__init__()
        self.method = method
        self.hidden_dim = hidden_dim

        if method == 'general':
            self.W_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif method == 'concat':
            self.W = nn.Linear(hidden_dim * 2, hidden_dim)
            self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [1, batch, hidden]  ← CURRENT state (after LSTM)
        # encoder_outputs: [src_len, batch, hidden]

        src_len = encoder_outputs.shape[0]
        batch_size = encoder_outputs.shape[1]

        if self.method == 'dot':
            # Dot product: h^T · s
            # encoder_outputs: [src_len, batch, hidden]
            # decoder_hidden: [1, batch, hidden]
            scores = torch.sum(encoder_outputs * decoder_hidden, dim=2)
            # scores: [src_len, batch]

        elif self.method == 'general':
            # General: h^T · W_a · s
            energy = self.W_a(decoder_hidden)  # [1, batch, hidden]
            scores = torch.sum(encoder_outputs * energy, dim=2)  # [src_len, batch]

        elif self.method == 'concat':
            # Concat: v^T · tanh(W · [h; s])
            # Repeat decoder_hidden for all source positions
            decoder_hidden = decoder_hidden.repeat(src_len, 1, 1)  # [src_len, batch, hidden]
            # Concatenate
            combined = torch.cat([encoder_outputs, decoder_hidden], dim=2)  # [src_len, batch, hidden*2]
            energy = torch.tanh(self.W(combined))  # [src_len, batch, hidden]
            scores = self.v(energy).squeeze(2)  # [src_len, batch]

        # Softmax over source positions
        attention_weights = F.softmax(scores, dim=0)  # [src_len, batch]

        # Context vector: weighted sum
        # attention_weights: [src_len, batch] → [src_len, batch, 1]
        # encoder_outputs: [src_len, batch, hidden]
        context = (attention_weights.unsqueeze(2) * encoder_outputs).sum(dim=0)
        # context: [batch, hidden]

        return context, attention_weights

In [ ]:
class LuongDecoder(nn.Module):
    """Luong decoder: LSTM first, then attention"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout, method='general'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.attention = LuongAttention(hidden_dim, method)
        
        # LSTM input: just embedding (NO context here, unlike Bahdanau!)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1)
        
        # Combine layer: [output; context] → combined
        self.W_c = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
        
        # Output layer
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch] (one token per sequence)
        # hidden: [1, batch, hidden]
        # cell: [1, batch, hidden]
        # encoder_outputs: [src_len, batch, hidden]
        
        # 1. Embed input
        input = input.unsqueeze(0)  # [batch] → [1, batch]
        embedded = self.dropout(self.embedding(input))  # [1, batch, emb_dim]
        
        # 2. LSTM step (BEFORE attention, this is the key difference!)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [1, batch, hidden]
        # hidden: [1, batch, hidden] ← NEW hidden state
        
        # 3. Compute attention using CURRENT (new) hidden state
        context, attention_weights = self.attention(hidden, encoder_outputs)
        # context: [batch, hidden]
        # attention_weights: [src_len, batch]
        
        # 4. Combine LSTM output + context
        output = output.squeeze(0)  # [batch, hidden]
        combined_input = torch.cat([output, context], dim=1)  # [batch, hidden*2]
        combined = torch.tanh(self.W_c(combined_input))  # [batch, hidden]
        
        # 5. Predict next token
        prediction = self.fc_out(combined)  # [batch, vocab_size]
        
        return prediction, hidden, cell, attention_weights

In [ ]:
class Seq2SeqLuong(nn.Module):
    """Complete Luong attention model"""
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch]
        # trg: [trg_len, batch]

        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features

        # Store outputs and attention weights
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        attentions = torch.zeros(trg_len, batch_size, src.shape[0]).to(self.device)

        # Encode source sequence
        encoder_outputs, hidden, cell = self.encoder(src)

        # First decoder input is <SOS>
        input = trg[0, :]  # [batch]

        # Decode one step at a time
        for t in range(1, trg_len):
            # Decode one step
            prediction, hidden, cell, attention_weights = self.decoder(
                input, hidden, cell, encoder_outputs
            )

            # Store prediction and attention
            outputs[t] = prediction
            attentions[t] = attention_weights.permute(1, 0)  # [batch, src_len]

            # Teacher forcing: use true token or predicted token
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)  # [batch]
            input = trg[t] if teacher_force else top1

        return outputs, attentions

## 6. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs, _ = model(src, trg)

        # Reshape for loss: skip <SOS> at position 0
        output_dim = outputs.shape[-1]
        outputs = outputs[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        loss = criterion(outputs, trg)
        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)

            # No teacher forcing
            outputs, _ = model(src, trg, teacher_forcing_ratio=0)

            output_dim = outputs.shape[-1]
            outputs = outputs[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(outputs, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

## 7. Train the Model

In [ ]:
# Initialize model
encoder = Encoder(src_vocab.n_words, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT)
decoder = LuongDecoder(trg_vocab.n_words, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT, method=ATTENTION_METHOD)
model = Seq2SeqLuong(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <PAD>

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model initialized with {total_params:,} trainable parameters')
print(f'Attention method: {ATTENTION_METHOD}\n')

# Training loop
train_losses = []
val_losses = []

print('Training started...\n')

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

print('\nTraining complete!')

## 8. Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Luong Attention Training Progress')
plt.legend()
plt.grid(True)
plt.show()

## 9. Test Translation

In [ ]:
def translate_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=50):
    """Translate a single sentence"""
    model.eval()
    
    # Tokenize and encode
    tokens = src_vocab.encode(sentence) + [src_vocab.word2idx['<EOS>']]
    src_tensor = torch.tensor(tokens).unsqueeze(1).to(device)  # [src_len, 1]
    
    with torch.no_grad():
        # Encode
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Start with <SOS>
        input = torch.tensor([trg_vocab.word2idx['<SOS>']]).to(device)
        
        translated = []
        attentions = []
        
        for _ in range(max_len):
            prediction, hidden, cell, attention_weights = model.decoder(
                input, hidden, cell, encoder_outputs
            )
            
            # Get predicted token
            top1 = prediction.argmax(1)
            predicted_token = top1.item()
            
            # Stop if <EOS>
            if predicted_token == trg_vocab.word2idx['<EOS>']:
                break
            
            translated.append(predicted_token)
            attentions.append(attention_weights.squeeze(1).cpu())
            
            # Next input
            input = top1
    
    # Decode translation
    translated_words = [trg_vocab.idx2word[idx] for idx in translated]
    
    return translated_words, attentions


# Test on validation examples
print('Translation Examples:\n')
for i in [0, 10, 50, 100]:
    src_sent = val_data[i]['en']
    trg_sent = val_data[i]['de']
    
    translation, _ = translate_sentence(model, src_sent, src_vocab, trg_vocab, device)
    
    print(f'Source:      {src_sent}')
    print(f'Target:      {trg_sent}')
    print(f'Predicted:   {" ".join(translation)}')
    print()

## 10. Visualize Attention Weights

In [ ]:
def plot_attention(sentence, translation, attentions):
    """Plot attention heatmap"""
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Convert to numpy
    attention_matrix = torch.stack(attentions).numpy()  # [trg_len, src_len]
    
    # Plot heatmap
    cax = ax.matshow(attention_matrix, cmap='viridis')
    fig.colorbar(cax)
    
    # Set ticks
    src_tokens = sentence.lower().split() + ['<EOS>']
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(translation)))
    
    ax.set_xticklabels(src_tokens, rotation=45)
    ax.set_yticklabels(translation)
    
    ax.set_xlabel('Source (English)')
    ax.set_ylabel('Target (German)')
    ax.set_title('Luong Attention Weights')
    
    plt.tight_layout()
    plt.show()


# Visualize attention for an example
example_idx = 10
src_sent = val_data[example_idx]['en']
translation, attentions = translate_sentence(model, src_sent, src_vocab, trg_vocab, device)

print(f'Source: {src_sent}')
print(f'Translation: {" ".join(translation)}\n')

if attentions:
    plot_attention(src_sent, translation, attentions)

## 11. Compare Different Attention Methods

In [ ]:
# Optional: Train models with different attention methods and compare
print('Comparison of attention methods:\n')
print('We trained with:', ATTENTION_METHOD)
print(f'Final validation loss: {val_losses[-1]:.3f}\n')

print('To compare methods, train separate models with:')
print('  - method="dot"     (fastest, simplest)')
print('  - method="general" (balanced, most popular)')
print('  - method="concat"  (most parameters, similar to Bahdanau)')

## Summary

### What we implemented:
1. **Luong Attention** with three scoring methods (dot/general/concat)
2. **Key difference from Bahdanau**: LSTM first, then attention
3. **Simpler architecture**: Uses current decoder state

### Key takeaways:
- Luong uses **current** hidden state s_t (after LSTM), Bahdanau uses **previous** s_t-1
- Context is combined with **LSTM output**, not input
- **General** scoring is most popular (good balance)
- **Dot** scoring is fastest but requires matching hidden sizes
- Often achieves similar or better performance than Bahdanau with fewer parameters

### Next steps:
- Try different attention methods (dot/general/concat)
- Compare with Bahdanau implementation
- Implement local attention for long sequences
- Move to Transformer attention (self-attention)